# IDM-VTON comparison on T4
Non-commercial benchmark only. Run after notebook 01 prepares the same images/masks/poses; preserve that prepared folder as a private dataset or keep it in this session. Runs in a separate process with CPU offload. GPU success and latency remain unverified. This uses shared CatVTON preprocessing, not IDM’s upper-body-only demo mask or a reproduction of paper metrics.

In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT = Path('/kaggle/working/fitsyncgemini')
# After merging the PR use main; before merging use the PR branch below.
BRANCH = 'codex/async-vton-prototype'
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/sadad54/fitsyncgemini.git',str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)


In [ ]:
VENV = Path('/kaggle/working/idm-env')
subprocess.run([sys.executable,'-m','uv','venv','--python','3.10',str(VENV)],check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([sys.executable,'-m','uv','pip','install','--python',PYTHON,'-r',str(ROOT/'gpu_worker/requirements-idm.txt')],check=True)
IDM = Path('/kaggle/working/IDM-VTON')
if not IDM.exists(): subprocess.run(['git','clone','https://github.com/yisol/IDM-VTON.git',str(IDM)],check=True)
subprocess.run(['git','-C',str(IDM),'checkout','0d5f3ec2d737487a9bb24e4100936ad254780383'],check=True)
os.environ['IDMVTON_DIR'] = str(IDM)
PREPARED = Path('/kaggle/working/vton-prepared/prepared.json')
RESULTS = Path('/kaggle/working/vton-results')
subprocess.run([PYTHON,str(ROOT/'gpu_worker/benchmark.py'),'idm',str(PREPARED),str(RESULTS)],check=True)


## Decide from evidence
Compare output pairs blindly. Fill the blank quality columns in each CSV (5 is best; for artifacts, 5 means none). Reject identity changes, missing limbs and wrong dress lengths. Compare cold/model-load time, preprocessing, first inference, later inference, and peak VRAM separately. Repeat at seed 7 and 123 before selecting a winner. CPU offload trades speed for memory. An out-of-memory/error row is a failed case, not a timing success. Do not infer a monthly free generation count from GPU-only seconds.